# **[_CREATING CUSTOMER TABLES_](url)**

In [0]:
# Store the values from the text input widgets into variables
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")

# Set path to your volume
my_volume_path = f"/Volumes/{my_catalog}/{my_schema}/retail_data"

# display the variables
print(f"my_catalog: {my_catalog}")
print(f"my_schema: {my_schema}")
print(f"my_volume_path: {my_volume_path}")

In [0]:
# 1. Drop table if exists 
spark.sql(f"DROP TABLE IF EXISTS {my_catalog}.{my_schema}.tb_customers_bronze")

# 2. Read data from volume
query = f"""
    CREATE OR REPLACE TABLE {my_catalog}.{my_schema}.tb_customers_bronze
    AS 
    SELECT 
        *,
        _metadata.file_path as source_file_path,
        _metadata.file_name as source_file_name,
        _metadata.file_size as source_file_size,
        _metadata.file_modification_time as source_file_modification_time,
        current_timestamp() as ingestion_timestamp
    FROM 
        read_files('{my_volume_path}/customers_*.csv',
            format => 'csv',
            header => 'true',
            delimiter => ',',
            inferSchema => 'true',
            ignoreLeadingWhiteSpace => 'true',
            ignoreTrailingWhiteSpace => 'true',
            dateFormat => 'yyyy-MM-dd HH:mm:ss'
        )  
    """

print(query)
spark.sql(query)